# Tesis — Pipeline V5 final reproducible

**Objetivo:** desarrollar y validar modelos de aprendizaje automático para estimar `Sa(T=1.0 s)` en sismos de subducción de interfaz de la placa de Nazca y realizar una aplicación paramétrica respecto de la E.030 vigente.

## Principios congelados

- Variable objetivo: `ln[Sa(T=1.0 s)]`, Sa RotD50, 5 % de amortiguamiento, unidades `g`.
- Dominio regional: Sudamérica, mecanismo de interfaz según `Intra_Inter_Flag == 0`.
- No se aplica un umbral adicional de profundidad hipocentral para definir interfaz.
- Evento único: `NGAsubEQID`.
- Estación única: `NGAsubSSN`.
- Pisco se mantiene fuera del desarrollo.
- El test interno se reserva antes de cualquier selección de algoritmo, variables o hiperparámetros.
- Validación agrupada por evento.
- Modelo final: Gradient Boosting.
- Variables finales: Mw, Rrup y Vs30.
- SHAP se interpreta como atribución predictiva, no causalidad.
- La aplicación a Lima es paramétrica y exploratoria, no una validación local.
- La comparación con E.030 es descriptiva; no se interpreta como seguridad, insuficiencia o conservadurismo normativo.

Este notebook está diseñado para ejecutarse **de arriba hacia abajo con “Run all”** en Google Colab.


## 0. Configuración, entorno y reproducibilidad


In [ ]:
# Si hace falta en Colab, descomentar:
# !pip -q install xgboost shap openpyxl

import os, json, time, warnings, itertools, platform
from collections import Counter

import numpy as np
import pandas as pd

import sklearn
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
import shap

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    print("No se detectó Google Colab; se continúa sin montar Drive.")

RUTA_DATOS = "/content/drive/MyDrive/Tesis/Datos"
RUTA_MEGA = os.path.join(
    RUTA_DATOS,
    "NGAsub_MegaFlatfile_RotD50_050_R211022_public_ Base de datos Sin Filtrar.xlsx"
)

RUTA_SALIDAS = os.path.join(RUTA_DATOS, "Resultados_V5")
os.makedirs(RUTA_SALIDAS, exist_ok=True)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", __import__("xgboost").__version__)
print("shap:", shap.__version__)
print("SEED:", SEED)


In [ ]:
def metricas_regresion(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    resid = y_pred - y_true

    out = {
        "R2": np.nan,
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "BIAS": resid.mean(),
        "ABS_BIAS": abs(resid.mean()),
        "SD_RESIDUAL": resid.std(ddof=1) if len(resid) > 1 else np.nan
    }

    if len(y_true) >= 2 and np.var(y_true) > 0:
        out["R2"] = r2_score(y_true, y_pred)

    return out


def export_csv(df, nombre):
    ruta = os.path.join(RUTA_SALIDAS, nombre)
    df.to_csv(ruta, index=False)
    print("Guardado:", ruta)
    return ruta


## 1. Constitución y auditoría del dataset analítico


In [ ]:
COLUMNAS = [
    "DatabaseRegion",
    "NGAsubEQID",
    "NGAsubSSN",
    "Earthquake_Name",
    "Earthquake_Magnitude",
    "Hypocenter_Depth_km",
    "Intra_Inter_Flag",
    "DbQualityFlag",
    "ClstD_km",
    "Vs30_Selected_for_Analysis_m_s",
    "T1pt000S",
]

if not os.path.exists(RUTA_MEGA):
    raise FileNotFoundError(
        f"No se encontró el MegaFlatfile en: {RUTA_MEGA}"
    )

df = pd.read_excel(
    RUTA_MEGA,
    sheet_name="NGAsub_MegaFlatfile_RotD50_050",
    usecols=COLUMNAS
)

print("NGA-Sub total:", len(df))


In [ ]:
vars_clave = [
    "Earthquake_Magnitude",
    "ClstD_km",
    "Hypocenter_Depth_km",
    "Vs30_Selected_for_Analysis_m_s",
    "T1pt000S"
]

df_sa = df[df["DatabaseRegion"] == "SouthAmerica"].copy()
df_interfaz = df_sa[df_sa["Intra_Inter_Flag"] == 0].copy()

df_completo = df_interfaz[
    (df_interfaz[vars_clave] > 0).all(axis=1) &
    (df_interfaz[vars_clave] != -999).all(axis=1)
].copy()

df_final = df_completo[
    df_completo["DbQualityFlag"] != -3
].copy()

auditoria = pd.DataFrame([
    ["NGA-Sub total", len(df), df["NGAsubEQID"].nunique(), df["NGAsubSSN"].nunique()],
    ["Sudamérica", len(df_sa), df_sa["NGAsubEQID"].nunique(), df_sa["NGAsubSSN"].nunique()],
    ["Interfaz", len(df_interfaz), df_interfaz["NGAsubEQID"].nunique(), df_interfaz["NGAsubSSN"].nunique()],
    ["Completitud", len(df_completo), df_completo["NGAsubEQID"].nunique(), df_completo["NGAsubSSN"].nunique()],
    ["Muestra final", len(df_final), df_final["NGAsubEQID"].nunique(), df_final["NGAsubSSN"].nunique()],
], columns=["Etapa","N_registros","N_eventos","N_estaciones"])

display(auditoria)

assert len(df) == 71340
assert len(df_sa) == 5994
assert len(df_interfaz) == 2080
assert len(df_final) == 1987
assert df_final["NGAsubEQID"].nunique() == 108
assert df_final["NGAsubSSN"].nunique() == 692

print("CONTROLES DEL DATASET: OK")
export_csv(auditoria, "01_auditoria_dataset.csv")
export_csv(df_final, "01_dataset_analitico_v5_1987.csv")


**Resultado esperado:** 1,987 registros, 108 eventos y 692 estaciones.

La profundidad hipocentral se conserva para auditoría y análisis comparativos, pero no se usa como filtro adicional para definir los registros de interfaz.


## 2. Pisco, desarrollo, test interno y folds externos


In [ ]:
pisco_mask = df_final["Earthquake_Name"].astype(str).str.contains(
    "Pisco", case=False, na=False
)

ids_pisco = df_final.loc[pisco_mask, "NGAsubEQID"].dropna().unique()
assert len(ids_pisco) == 1

PISCO_EQID = ids_pisco[0]

df_pisco = df_final[df_final["NGAsubEQID"] == PISCO_EQID].copy()
df_pool = df_final[df_final["NGAsubEQID"] != PISCO_EQID].copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

idx_dev, idx_test = next(
    gss.split(df_pool, groups=df_pool["NGAsubEQID"])
)

df_dev = df_pool.iloc[idx_dev].copy().reset_index(drop=True)
df_test = df_pool.iloc[idx_test].copy().reset_index(drop=True)

assert set(df_dev["NGAsubEQID"]).isdisjoint(set(df_test["NGAsubEQID"]))
assert PISCO_EQID not in set(df_dev["NGAsubEQID"])
assert PISCO_EQID not in set(df_test["NGAsubEQID"])

resumen_particiones = pd.DataFrame([
    ["DESARROLLO", len(df_dev), df_dev["NGAsubEQID"].nunique(), df_dev["NGAsubSSN"].nunique()],
    ["TEST_INTERNO", len(df_test), df_test["NGAsubEQID"].nunique(), df_test["NGAsubSSN"].nunique()],
    ["PISCO_HOLDOUT", len(df_pisco), df_pisco["NGAsubEQID"].nunique(), df_pisco["NGAsubSSN"].nunique()],
], columns=["Particion","N_registros","N_eventos","N_estaciones"])

display(resumen_particiones)
export_csv(resumen_particiones, "02_resumen_particiones.csv")


In [ ]:
outer_cv = GroupKFold(n_splits=5)

df_dev["outer_fold"] = np.nan
for fold, (_, idx_val) in enumerate(
    outer_cv.split(df_dev, groups=df_dev["NGAsubEQID"]),
    start=1
):
    df_dev.loc[df_dev.index[idx_val], "outer_fold"] = fold

df_dev["outer_fold"] = df_dev["outer_fold"].astype(int)

assert (df_dev.groupby("NGAsubEQID")["outer_fold"].nunique() == 1).all()

display(
    df_dev.groupby("outer_fold").agg(
        N_registros=("NGAsubEQID","size"),
        N_eventos=("NGAsubEQID","nunique")
    )
)


## 3. Benchmark anidado de ocho algoritmos

Para reproducir exactamente los resultados reportados en la tesis, este bloque conserva el mismo número de configuraciones aleatorias usado en la corrida V5 consolidada.


In [ ]:
FEATURE_MAP_4 = {
    "Mw":"Earthquake_Magnitude",
    "Rrup":"ClstD_km",
    "Zhyp":"Hypocenter_Depth_km",
    "Vs30":"Vs30_Selected_for_Analysis_m_s"
}

X_dev4 = pd.DataFrame({
    k: df_dev[v].to_numpy()
    for k,v in FEATURE_MAP_4.items()
})
y_dev = np.log(df_dev["T1pt000S"].to_numpy())
groups_dev = df_dev["NGAsubEQID"].to_numpy()

modelos = {
    "Linear Regression": {
        "estimator": LinearRegression(),
        "params": {"fit_intercept":[True,False]},
        "search":"grid"
    },
    "ElasticNet": {
        "estimator": Pipeline([
            ("scaler",StandardScaler()),
            ("model",ElasticNet(max_iter=20000,random_state=SEED))
        ]),
        "params":{
            "model__alpha":np.logspace(-4,0,9),
            "model__l1_ratio":[0.1,0.3,0.5,0.7,0.9]
        },
        "search":"random",
        "n_iter":15
    },
    "KNN": {
        "estimator": Pipeline([
            ("scaler",StandardScaler()),
            ("model",KNeighborsRegressor())
        ]),
        "params":{
            "model__n_neighbors":list(range(5,31)),
            "model__weights":["uniform","distance"],
            "model__p":[1,2]
        },
        "search":"random",
        "n_iter":15
    },
    "SVR": {
        "estimator": Pipeline([
            ("scaler",StandardScaler()),
            ("model",SVR())
        ]),
        "params":{
            "model__C":np.logspace(0,2,10),
            "model__gamma":["scale",0.01,0.03,0.1,0.3],
            "model__epsilon":[0.05,0.10,0.20,0.30]
        },
        "search":"random",
        "n_iter":20
    },
    "Random Forest": {
        "estimator": RandomForestRegressor(random_state=SEED,n_jobs=1),
        "params":{
            "n_estimators":[200,300,400],
            "max_depth":[None,8,12,16],
            "min_samples_split":[2,5,10],
            "min_samples_leaf":[1,2,4],
            "max_features":[0.50,0.75,1.00],
            "bootstrap":[True,False]
        },
        "search":"random",
        "n_iter":10
    },
    "Extra Trees": {
        "estimator": ExtraTreesRegressor(random_state=SEED,n_jobs=1),
        "params":{
            "n_estimators":[200,300,400],
            "max_depth":[None,8,12,16],
            "min_samples_split":[2,5,10],
            "min_samples_leaf":[1,2,4],
            "max_features":[0.50,0.75,1.00],
            "bootstrap":[True,False]
        },
        "search":"random",
        "n_iter":10
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingRegressor(random_state=SEED),
        "params":{
            "n_estimators":[100,200,300,400],
            "learning_rate":[0.02,0.05,0.10],
            "max_depth":[2,3,4],
            "subsample":[0.70,0.85,1.00],
            "min_samples_split":[2,5,10],
            "min_samples_leaf":[1,2,4]
        },
        "search":"random",
        "n_iter":10
    },
    "XGBoost": {
        "estimator": XGBRegressor(
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=1,
            tree_method="hist"
        ),
        "params":{
            "n_estimators":[200,300,400],
            "max_depth":[2,3,4,5],
            "learning_rate":[0.02,0.05,0.10],
            "subsample":[0.70,0.85,1.00],
            "colsample_bytree":[0.70,0.85,1.00],
            "reg_alpha":[0.0,0.01,0.10,1.0],
            "reg_lambda":[0.5,1.0,2.0,5.0]
        },
        "search":"random",
        "n_iter":10
    }
}

print("Modelos:", list(modelos))


In [ ]:
pred_oof = {}
fold_rows = []
best_params_nested = {}

# baseline ingenuo
pred_naive = np.full(len(df_dev), np.nan)
for tr, va in outer_cv.split(X_dev4, y_dev, groups_dev):
    pred_naive[va] = y_dev[tr].mean()

pred_oof["Naive Mean"] = pred_naive

for nombre, cfg in modelos.items():
    print("\n===", nombre, "===")
    oof = np.full(len(df_dev), np.nan)
    params_folds = []

    for fold, (tr, va) in enumerate(
        outer_cv.split(X_dev4, y_dev, groups_dev),
        start=1
    ):
        inner = GroupKFold(n_splits=5)

        if cfg["search"] == "grid":
            search = GridSearchCV(
                clone(cfg["estimator"]),
                cfg["params"],
                scoring="r2",
                cv=inner,
                n_jobs=-1,
                refit=True
            )
        else:
            search = RandomizedSearchCV(
                clone(cfg["estimator"]),
                cfg["params"],
                n_iter=cfg["n_iter"],
                scoring="r2",
                cv=inner,
                random_state=SEED,
                n_jobs=-1,
                refit=True
            )

        search.fit(
            X_dev4.iloc[tr],
            y_dev[tr],
            groups=groups_dev[tr]
        )

        pred = search.best_estimator_.predict(X_dev4.iloc[va])
        oof[va] = pred
        params_folds.append(search.best_params_)

        m = metricas_regresion(y_dev[va], pred)
        fold_rows.append({
            "Modelo":nombre,
            "Fold":fold,
            **m
        })

        print(
            f"Fold {fold}: R2={m['R2']:.4f}, "
            f"RMSE={m['RMSE']:.4f}, MAE={m['MAE']:.4f}"
        )

    pred_oof[nombre] = oof
    best_params_nested[nombre] = params_folds

fold_df = pd.DataFrame(fold_rows)

global_rows = []
for nombre,pred in pred_oof.items():
    m = metricas_regresion(y_dev,pred)
    sd = (
        fold_df.loc[fold_df["Modelo"]==nombre,"R2"].std(ddof=1)
        if nombre != "Naive Mean" else np.nan
    )
    global_rows.append({
        "Modelo":nombre,
        **m,
        "R2_FOLD_SD":sd
    })

benchmark = pd.DataFrame(global_rows).sort_values("RMSE")
display(benchmark)

export_csv(benchmark, "03_benchmark_nested_oof.csv")
export_csv(fold_df, "03_benchmark_nested_folds.csv")


In [ ]:
ranking = benchmark[benchmark["Modelo"]!="Naive Mean"].copy()

for col, asc in [
    ("R2",False),
    ("RMSE",True),
    ("MAE",True),
    ("ABS_BIAS",True),
    ("SD_RESIDUAL",True),
    ("R2_FOLD_SD",True)
]:
    ranking["rank_"+col] = ranking[col].rank(
        ascending=asc,
        method="average"
    )

rank_cols = [
    "rank_R2","rank_RMSE","rank_MAE",
    "rank_ABS_BIAS","rank_SD_RESIDUAL","rank_R2_FOLD_SD"
]

ranking["SUMA_RANGOS"] = ranking[rank_cols].sum(axis=1)
ranking = ranking.sort_values(
    ["SUMA_RANGOS","RMSE","MAE"]
).reset_index(drop=True)

display(ranking[
    ["Modelo","R2","RMSE","MAE","BIAS","R2_FOLD_SD","SUMA_RANGOS"]
])

export_csv(ranking, "03_ranking_multicriterio.csv")


### 3.1. Bootstrap pareado por evento entre modelos

Esta prueba utiliza las predicciones OOF ya generadas en el benchmark. No modifica la selección del algoritmo. Su finalidad es cuantificar si las diferencias de R² entre Gradient Boosting y los demás modelos son robustas al remuestreo de eventos completos.


In [ ]:
rng = np.random.default_rng(SEED)
event_ids = df_dev["NGAsubEQID"].to_numpy()
unique_events = np.unique(event_ids)
idx_by_event = {
    ev: np.flatnonzero(event_ids == ev)
    for ev in unique_events
}

def bootstrap_delta_r2(pred_a, pred_b, n_boot=3000):
    deltas = []
    for _ in range(n_boot):
        sampled = rng.choice(
            unique_events,
            size=len(unique_events),
            replace=True
        )
        idx = np.concatenate([
            idx_by_event[e] for e in sampled
        ])
        r2a = r2_score(y_dev[idx], pred_a[idx])
        r2b = r2_score(y_dev[idx], pred_b[idx])
        deltas.append(r2a - r2b)

    deltas = np.asarray(deltas)
    return {
        "Delta_R2_mediana": np.median(deltas),
        "IC95_inf": np.quantile(deltas, 0.025),
        "IC95_sup": np.quantile(deltas, 0.975)
    }

bootstrap_models = []
gb_pred = pred_oof["Gradient Boosting"]

for other in [
    "SVR", "Random Forest", "Extra Trees", "XGBoost",
    "KNN", "ElasticNet", "Linear Regression"
]:
    res = bootstrap_delta_r2(
        gb_pred,
        pred_oof[other],
        n_boot=3000
    )
    bootstrap_models.append({
        "Comparacion": f"Gradient Boosting - {other}",
        **res
    })

bootstrap_models = pd.DataFrame(bootstrap_models)
display(bootstrap_models)
export_csv(
    bootstrap_models,
    "03_bootstrap_pareado_modelos_evento.csv"
)


## 4. Dependencia entre predictores y selección de variables


In [ ]:
pearson = X_dev4.corr(method="pearson")
spearman = X_dev4.corr(method="spearman")

vif_rows=[]
for target in X_dev4.columns:
    others=[c for c in X_dev4.columns if c != target]
    reg=LinearRegression().fit(X_dev4[others],X_dev4[target])
    r2_aux=reg.score(X_dev4[others],X_dev4[target])
    vif_rows.append({
        "Variable":target,
        "R2_auxiliar":r2_aux,
        "VIF":1/(1-r2_aux)
    })

vif = pd.DataFrame(vif_rows)

display(pearson)
display(spearman)
display(vif)

pearson.to_csv(os.path.join(RUTA_SALIDAS,"04_pearson_predictores.csv"))
spearman.to_csv(os.path.join(RUTA_SALIDAS,"04_spearman_predictores.csv"))
export_csv(vif,"04_vif_predictores.csv")


In [ ]:
GB_REF = {
    "learning_rate":0.10,
    "max_depth":3,
    "min_samples_leaf":4,
    "min_samples_split":10,
    "n_estimators":200,
    "subsample":0.70
}

feature_map = FEATURE_MAP_4.copy()
variables = ["Mw","Rrup","Zhyp","Vs30"]

combo_rows=[]
combo_oof={}

for r in range(1,5):
    for combo in itertools.combinations(variables,r):
        cols=[feature_map[v] for v in combo]
        Xc=df_dev[cols].copy()
        Xc.columns=list(combo)

        oof=np.full(len(df_dev),np.nan)
        r2_folds=[]

        for tr,va in outer_cv.split(Xc,y_dev,groups_dev):
            mod=GradientBoostingRegressor(
                random_state=SEED,
                **GB_REF
            )
            mod.fit(Xc.iloc[tr],y_dev[tr])
            pred=mod.predict(Xc.iloc[va])
            oof[va]=pred
            r2_folds.append(r2_score(y_dev[va],pred))

        m=metricas_regresion(y_dev,oof)

        combo_name="+".join(combo)
        combo_rows.append({
            "Combinacion":combo_name,
            "N_variables":len(combo),
            **m,
            "R2_FOLD_SD":np.std(r2_folds,ddof=1)
        })
        combo_oof[combo_name]=oof

tabla_combos=pd.DataFrame(combo_rows).sort_values(
    ["R2","RMSE","MAE"],
    ascending=[False,True,True]
).reset_index(drop=True)

display(tabla_combos)
export_csv(tabla_combos,"04_seleccion_15_combinaciones.csv")


### 4.1. Bootstrap del subconjunto seleccionado frente al modelo de cuatro variables

La comparación se efectúa con las predicciones OOF obtenidas bajo la misma configuración de Gradient Boosting. El objetivo es verificar si la ventaja del subconjunto de tres predictores es robusta o si ambos modelos presentan desempeño estadísticamente comparable.


In [ ]:
rng_combo = np.random.default_rng(SEED)
pred_3 = combo_oof["Mw+Rrup+Vs30"]
pred_4 = combo_oof["Mw+Rrup+Zhyp+Vs30"]

deltas_r2 = []
deltas_rmse = []
deltas_mae = []

for _ in range(3000):
    sampled = rng_combo.choice(
        unique_events,
        size=len(unique_events),
        replace=True
    )
    idx = np.concatenate([
        idx_by_event[e] for e in sampled
    ])

    r2_3 = r2_score(y_dev[idx], pred_3[idx])
    r2_4 = r2_score(y_dev[idx], pred_4[idx])

    rmse_3 = np.sqrt(mean_squared_error(y_dev[idx], pred_3[idx]))
    rmse_4 = np.sqrt(mean_squared_error(y_dev[idx], pred_4[idx]))

    mae_3 = mean_absolute_error(y_dev[idx], pred_3[idx])
    mae_4 = mean_absolute_error(y_dev[idx], pred_4[idx])

    deltas_r2.append(r2_3-r2_4)
    deltas_rmse.append(rmse_3-rmse_4)
    deltas_mae.append(mae_3-mae_4)

def qsum(x):
    x=np.asarray(x)
    return np.median(x), np.quantile(x,.025), np.quantile(x,.975)

r2m,r2lo,r2hi=qsum(deltas_r2)
rmm,rmlo,rmhi=qsum(deltas_rmse)
mam,malo,mahi=qsum(deltas_mae)

bootstrap_combo = pd.DataFrame([
    ["Delta_R2", r2m, r2lo, r2hi],
    ["Delta_RMSE", rmm, rmlo, rmhi],
    ["Delta_MAE", mam, malo, mahi],
], columns=["Metrica","Mediana","IC95_inf","IC95_sup"])

display(bootstrap_combo)
export_csv(
    bootstrap_combo,
    "04_bootstrap_3vars_vs_4vars.csv"
)


**Decisión congelada:** `Mw + Rrup + Vs30`.

`Zhyp` se excluye por parsimonia al no aportar una mejora robusta de generalización frente al modelo de cuatro variables. No se interpreta como irrelevancia física.


## 5. Congelamiento del modelo final y apertura única del test


In [ ]:
FEATURES_FINAL = [
    "Earthquake_Magnitude",
    "ClstD_km",
    "Vs30_Selected_for_Analysis_m_s"
]

X_dev_final = df_dev[FEATURES_FINAL].copy()
X_test_final = df_test[FEATURES_FINAL].copy()

y_dev_final = np.log(df_dev["T1pt000S"].to_numpy())
y_test_final = np.log(df_test["T1pt000S"].to_numpy())

param_space_final = {
    "n_estimators":[100,150,200,250,300,400,500],
    "learning_rate":[0.01,0.02,0.03,0.05,0.075,0.10],
    "max_depth":[2,3,4],
    "subsample":[0.65,0.70,0.80,0.90,1.00],
    "min_samples_split":[2,5,10,15],
    "min_samples_leaf":[1,2,4,6]
}

search_final = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=SEED),
    param_space_final,
    n_iter=25,
    scoring="r2",
    cv=GroupKFold(n_splits=5),
    random_state=SEED,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

search_final.fit(
    X_dev_final,
    y_dev_final,
    groups=df_dev["NGAsubEQID"].to_numpy()
)

FINAL_PARAMS = search_final.best_params_
print("Hiperparámetros finales:", FINAL_PARAMS)
print("R2 medio CV desarrollo:", round(search_final.best_score_,4))

with open(
    os.path.join(RUTA_SALIDAS,"05_hiperparametros_finales.json"),
    "w"
) as f:
    json.dump(FINAL_PARAMS,f,indent=2)


In [ ]:
modelo_final = GradientBoostingRegressor(
    random_state=SEED,
    **FINAL_PARAMS
)
modelo_final.fit(X_dev_final,y_dev_final)

pred_test = modelo_final.predict(X_test_final)
m_test = metricas_regresion(y_test_final,pred_test)

display(pd.DataFrame([m_test]))

assert round(m_test["R2"],4) == 0.8728
assert round(m_test["RMSE"],4) == 1.1516

test_out=df_test[
    [
        "NGAsubEQID","NGAsubSSN","Earthquake_Name",
        "Earthquake_Magnitude","ClstD_km",
        "Vs30_Selected_for_Analysis_m_s","T1pt000S"
    ]
].copy()

test_out["lnSa_obs"]=y_test_final
test_out["lnSa_pred"]=pred_test
test_out["residual_pred_minus_obs"]=pred_test-y_test_final
test_out["Sa_obs_g"]=np.exp(y_test_final)
test_out["Sa_pred_g"]=np.exp(pred_test)

export_csv(test_out,"05_test_predicciones_finales.csv")
export_csv(pd.DataFrame([m_test]),"05_test_metricas_finales.csv")


### 5.1. Diagnóstico por evento, baselines y bootstrap del test final

Estas pruebas utilizan únicamente las predicciones ya congeladas del test. No se emplean para volver a seleccionar el modelo.


In [ ]:
# Métricas por evento
test_event_rows = []
for ev, g in test_out.groupby("NGAsubEQID"):
    mt = metricas_regresion(
        g["lnSa_obs"].to_numpy(),
        g["lnSa_pred"].to_numpy()
    )
    test_event_rows.append({
        "NGAsubEQID": ev,
        "Earthquake_Name": g["Earthquake_Name"].iloc[0],
        "N_registros": len(g),
        **mt
    })

test_event = pd.DataFrame(test_event_rows)

event_equal_weight = pd.DataFrame([{
    "RMSE_event_weighted": test_event["RMSE"].mean(),
    "MAE_event_weighted": test_event["MAE"].mean(),
    "BIAS_event_weighted": test_event["BIAS"].mean(),
    "ABS_BIAS_event_weighted": test_event["ABS_BIAS"].mean(),
    "Mediana_R2_evento": test_event["R2"].median()
}])

display(event_equal_weight)

export_csv(
    test_event,
    "05_test_metricas_por_evento.csv"
)
export_csv(
    event_equal_weight,
    "05_test_metricas_equal_weight_evento.csv"
)

# Baselines bajo exactamente el mismo test
lin = LinearRegression().fit(
    X_dev_final,
    y_dev_final
)
pred_lin = lin.predict(X_test_final)
pred_naive_test = np.full(
    len(y_test_final),
    y_dev_final.mean()
)

baseline_test = pd.DataFrame([
    {"Modelo":"Gradient Boosting", **metricas_regresion(y_test_final,pred_test)},
    {"Modelo":"Linear Regression", **metricas_regresion(y_test_final,pred_lin)},
    {"Modelo":"Naive Mean", **metricas_regresion(y_test_final,pred_naive_test)},
])

display(baseline_test)
export_csv(
    baseline_test,
    "05_test_comparacion_baselines.csv"
)

# Bootstrap por evento del test
rng_test=np.random.default_rng(SEED)
test_events=df_test["NGAsubEQID"].unique()
test_idx={
    ev:np.flatnonzero(df_test["NGAsubEQID"].to_numpy()==ev)
    for ev in test_events
}

boot=[]
for _ in range(5000):
    sampled=rng_test.choice(
        test_events,
        size=len(test_events),
        replace=True
    )
    idx=np.concatenate([test_idx[e] for e in sampled])
    boot.append(
        metricas_regresion(
            y_test_final[idx],
            pred_test[idx]
        )
    )

boot_df=pd.DataFrame(boot)
boot_summary=[]
for col in ["R2","RMSE","MAE","BIAS","SD_RESIDUAL"]:
    boot_summary.append({
        "Metrica":col,
        "Mediana":boot_df[col].median(),
        "IC95_inf":boot_df[col].quantile(.025),
        "IC95_sup":boot_df[col].quantile(.975)
    })

boot_summary=pd.DataFrame(boot_summary)
display(boot_summary)
export_csv(
    boot_summary,
    "05_test_bootstrap_eventos.csv"
)


## 6. Generalización: Pisco, purga de estaciones y LOEO


In [ ]:
# Pisco
y_pisco=np.log(df_pisco["T1pt000S"].to_numpy())
pred_pisco=modelo_final.predict(df_pisco[FEATURES_FINAL])
m_pisco=metricas_regresion(y_pisco,pred_pisco)

print("Pisco:",m_pisco)

# Test con purga de estaciones
test_stations=set(df_test["NGAsubSSN"])
dev_purged=df_dev[
    ~df_dev["NGAsubSSN"].isin(test_stations)
].copy()

mod_purged=GradientBoostingRegressor(
    random_state=SEED,
    **FINAL_PARAMS
)
mod_purged.fit(
    dev_purged[FEATURES_FINAL],
    np.log(dev_purged["T1pt000S"])
)

pred_test_purged=mod_purged.predict(df_test[FEATURES_FINAL])
m_test_purged=metricas_regresion(
    y_test_final,
    pred_test_purged
)

print("Test con purga de estaciones:",m_test_purged)


### 6.1. Sensibilidad de Pisco a estaciones previamente observadas

Se eliminan del desarrollo todas las estaciones que aparecen en Pisco y se vuelve a ajustar el mismo modelo con hiperparámetros congelados. Esta es una prueba de sensibilidad, no una nueva selección de modelo.


In [ ]:
pisco_stations=set(df_pisco["NGAsubSSN"])

dev_pisco_purged = df_dev[
    ~df_dev["NGAsubSSN"].isin(pisco_stations)
].copy()

mod_pisco_purged = GradientBoostingRegressor(
    random_state=SEED,
    **FINAL_PARAMS
)
mod_pisco_purged.fit(
    dev_pisco_purged[FEATURES_FINAL],
    np.log(dev_pisco_purged["T1pt000S"])
)

pred_pisco_purged = mod_pisco_purged.predict(
    df_pisco[FEATURES_FINAL]
)

m_pisco_purged = metricas_regresion(
    y_pisco,
    pred_pisco_purged
)

pisco_sens = pd.DataFrame([
    {"Escenario":"Pisco normal", **m_pisco},
    {"Escenario":"Pisco + purga estaciones", **m_pisco_purged}
])

display(pisco_sens)
export_csv(
    pisco_sens,
    "06_pisco_sensibilidad_estaciones.csv"
)


In [ ]:
# LOEO estándar sobre pool no-Pisco
pool=pd.concat([df_dev,df_test],ignore_index=True)
events=pool["NGAsubEQID"].unique()

y_pool=np.log(pool["T1pt000S"].to_numpy())
pred_loeo=np.full(len(pool),np.nan)
rows_loeo=[]

for ev in events:
    mask=pool["NGAsubEQID"].eq(ev)
    tr=pool[~mask]
    va=pool[mask]

    mod=GradientBoostingRegressor(
        random_state=SEED,
        **FINAL_PARAMS
    )
    mod.fit(
        tr[FEATURES_FINAL],
        np.log(tr["T1pt000S"])
    )

    pv=mod.predict(va[FEATURES_FINAL])
    yv=np.log(va["T1pt000S"].to_numpy())
    pred_loeo[np.flatnonzero(mask.to_numpy())]=pv

    rows_loeo.append({
        "NGAsubEQID":ev,
        "Earthquake_Name":va["Earthquake_Name"].iloc[0],
        "N_registros":len(va),
        **metricas_regresion(yv,pv)
    })

loeo_eventos=pd.DataFrame(rows_loeo)
m_loeo=metricas_regresion(y_pool,pred_loeo)

print("LOEO global:",m_loeo)
print("Mediana R2 por evento:",loeo_eventos["R2"].median())

export_csv(loeo_eventos,"06_loeo_metricas_evento.csv")


In [ ]:
# LOEO + purga de estaciones
pred_loeo_purge=np.full(len(pool),np.nan)
rows_purge=[]

for ev in events:
    mask=pool["NGAsubEQID"].eq(ev)
    va=pool[mask]
    stations=set(va["NGAsubSSN"])

    tr=pool[
        (~mask) &
        (~pool["NGAsubSSN"].isin(stations))
    ].copy()

    mod=GradientBoostingRegressor(
        random_state=SEED,
        **FINAL_PARAMS
    )
    mod.fit(
        tr[FEATURES_FINAL],
        np.log(tr["T1pt000S"])
    )

    pv=mod.predict(va[FEATURES_FINAL])
    yv=np.log(va["T1pt000S"].to_numpy())
    pred_loeo_purge[np.flatnonzero(mask.to_numpy())]=pv

    rows_purge.append({
        "NGAsubEQID":ev,
        "Earthquake_Name":va["Earthquake_Name"].iloc[0],
        "N_registros":len(va),
        "N_train_purgado":len(tr),
        **metricas_regresion(yv,pv)
    })

loeo_purge_eventos=pd.DataFrame(rows_purge)
m_loeo_purge=metricas_regresion(y_pool,pred_loeo_purge)

print("LOEO + purga estaciones:",m_loeo_purge)
print(
    "Mediana R2 por evento:",
    loeo_purge_eventos["R2"].median()
)

export_csv(
    loeo_purge_eventos,
    "06_loeo_purga_estaciones_metricas_evento.csv"
)


## 7. Incertidumbre predictiva: envolvente empírica de error


In [ ]:
# Para reproducir la envolvente se genera OOF del modelo final
oof_final=np.full(len(df_dev),np.nan)

for tr,va in outer_cv.split(
    X_dev_final,
    y_dev_final,
    df_dev["NGAsubEQID"]
):
    mod=GradientBoostingRegressor(
        random_state=SEED,
        **FINAL_PARAMS
    )
    mod.fit(
        X_dev_final.iloc[tr],
        y_dev_final[tr]
    )
    oof_final[va]=mod.predict(X_dev_final.iloc[va])

resid=oof_final-y_dev_final

levels={
    "80%":(0.10,0.90),
    "90%":(0.05,0.95),
    "95%":(0.025,0.975)
}

env_rows=[]
for label,(a,b) in levels.items():
    qlo,qhi=np.quantile(resid,[a,b])
    env_rows.append({
        "Cobertura_nominal":label,
        "q_resid_inf":qlo,
        "q_resid_sup":qhi,
        "Factor_inferior_sobre_Sa_pred":np.exp(-qhi),
        "Factor_superior_sobre_Sa_pred":np.exp(-qlo)
    })

envolvente=pd.DataFrame(env_rows)
display(envolvente)
export_csv(envolvente,"07_envolvente_empirica_error.csv")


### 7.1. Cobertura observada de la envolvente en el test

La cobertura del test se usa como diagnóstico descriptivo de calibración empírica. No convierte la envolvente en un intervalo probabilístico universal.


In [ ]:
test_resid = pred_test - y_test_final

coverage_rows=[]
for _,r in envolvente.iterrows():
    qlo=r["q_resid_inf"]
    qhi=r["q_resid_sup"]
    coverage_rows.append({
        "Cobertura_nominal":r["Cobertura_nominal"],
        "Cobertura_test_observada":(
            (test_resid>=qlo)&(test_resid<=qhi)
        ).mean()
    })

coverage_test=pd.DataFrame(coverage_rows)
display(coverage_test)

export_csv(
    coverage_test,
    "07_cobertura_envolvente_test.csv"
)


## 8. SHAP e interacciones predictivas


In [ ]:
explainer=shap.TreeExplainer(modelo_final)

# SHAP se calcula sobre el desarrollo
X_shap=X_dev_final.copy()
X_shap.columns=["Mw","Rrup","Vs30"]

shap_values=explainer.shap_values(X_shap)

imp=pd.DataFrame({
    "Variable":X_shap.columns,
    "mean_abs_SHAP":np.mean(np.abs(shap_values),axis=0)
}).sort_values("mean_abs_SHAP",ascending=False)

interaction_values=np.asarray(
    explainer.shap_interaction_values(X_shap)
)

mean_abs_inter=np.mean(
    np.abs(interaction_values),
    axis=0
)

pairs=[]
for i in range(len(X_shap.columns)):
    for j in range(i+1,len(X_shap.columns)):
        pairs.append({
            "Variable_1":X_shap.columns[i],
            "Variable_2":X_shap.columns[j],
            "mean_abs_SHAP_interaction":mean_abs_inter[i,j]
        })

pairs_df=pd.DataFrame(pairs).sort_values(
    "mean_abs_SHAP_interaction",
    ascending=False
)

display(imp)
display(pairs_df)

export_csv(imp,"08_shap_importancia_global.csv")
export_csv(pairs_df,"08_shap_interacciones_pares.csv")


## 9. Dominio para Lima y comparación paramétrica con E.030-2026


In [ ]:
joint=df_final[
    df_final["Earthquake_Magnitude"].between(7.5,8.5) &
    df_final["ClstD_km"].between(50,150)
].copy()

print(
    "Ventana conjunta:",
    len(joint),"registros,",
    joint["NGAsubEQID"].nunique(),"eventos,",
    joint["NGAsubSSN"].nunique(),"estaciones"
)


In [ ]:
vs_scenarios=[650,450,250,150]
support_rows=[]

for vs in vs_scenarios:
    for tol in [25,50,100]:
        d=joint[
            joint["Vs30_Selected_for_Analysis_m_s"].between(
                vs-tol,vs+tol
            )
        ]
        support_rows.append({
            "Vs30_escenario":vs,
            "Tolerancia_m_s":tol,
            "N_registros_joint":len(d),
            "N_eventos_joint":d["NGAsubEQID"].nunique(),
            "N_estaciones_joint":d["NGAsubSSN"].nunique()
        })

support=pd.DataFrame(support_rows)
display(support)

export_csv(support,"09_soporte_vs30_lima.csv")


In [ ]:
# E.030-2026: referencia paramétrica Zona 4, T=1.0 s, U=1, R=1

def e030_zone4_params(vs30):
    if vs30 >= 800:
        return {"Perfil":"S0","S":0.80,"TP":0.30,"TL":3.00,"generica":True}
    elif vs30 >= 550:
        return {"Perfil":"S1","S":1.00,"TP":0.40,"TL":2.50,"generica":True}
    elif vs30 >= 350:
        f=(550-vs30)/(550-350)
        return {
            "Perfil":"S2",
            "S":1.00+f*(1.10-1.00),
            "TP":0.40+f*(0.60-0.40),
            "TL":2.50+f*(2.00-2.50),
            "generica":True
        }
    elif vs30 >= 200:
        f=(350-vs30)/(350-200)
        return {
            "Perfil":"S3",
            "S":1.10+f*(1.20-1.10),
            "TP":0.60+f*(0.90-0.60),
            "TL":2.00+f*(1.60-2.00),
            "generica":True
        }
    else:
        return {
            "Perfil":"S4",
            "S":np.nan,
            "TP":1.20,
            "TL":1.60,
            "generica":False
        }


def C_2026(T,TP,TL):
    if T < 0.2*TP:
        return 1 + 7.5*T/TP
    elif T <= TP:
        return 2.5
    elif T < TL:
        return 2.5*TP/T
    else:
        return 2.5*TP*TL/(T**2)


Z=0.45
T=1.0
U=1.0
R=1.0

norm_rows=[]
for vs in vs_scenarios:
    p=e030_zone4_params(vs)
    C=C_2026(T,p["TP"],p["TL"])

    Sa=(
        Z*p["S"]*C*U/R
        if p["generica"]
        else np.nan
    )

    norm_rows.append({
        "Vs30_m_s":vs,
        "Perfil_E030_2026":p["Perfil"],
        "S":p["S"],
        "TP_s":p["TP"],
        "TL_s":p["TL"],
        "C_T1s":C,
        "Sa_E030_T1_g":Sa,
        "Comparacion_generica":p["generica"]
    })

norm=pd.DataFrame(norm_rows)
display(norm)

export_csv(norm,"09_e030_2026_parametros_T1_zona4.csv")


In [ ]:
mw_grid=[7.5,8.0,8.5]
rrup_grid=[50,75,100,125,150]

rows=[]
for vs in vs_scenarios:
    for mw in mw_grid:
        for rr in rrup_grid:
            Xnew=pd.DataFrame([{
                "Earthquake_Magnitude":mw,
                "ClstD_km":rr,
                "Vs30_Selected_for_Analysis_m_s":vs
            }])

            lnsa=float(modelo_final.predict(Xnew)[0])
            rows.append({
                "Mw":mw,
                "Rrup_km":rr,
                "Vs30_m_s":vs,
                "lnSa_modelo":lnsa,
                "Sa_modelo_g":np.exp(lnsa)
            })

escenarios=pd.DataFrame(rows).merge(
    norm[[
        "Vs30_m_s",
        "Perfil_E030_2026",
        "Sa_E030_T1_g",
        "Comparacion_generica"
    ]],
    on="Vs30_m_s",
    how="left"
)

escenarios["ratio_modelo_E030"] = (
    escenarios["Sa_modelo_g"] /
    escenarios["Sa_E030_T1_g"]
)

display(escenarios)
export_csv(escenarios,"09_escenarios_lima_modelo_e030.csv")


## 10. Resumen reproducible de resultados y controles finales


In [ ]:
resumen_final = pd.DataFrame([
    ["Dataset final", len(df_final), np.nan],
    ["Eventos finales", df_final["NGAsubEQID"].nunique(), np.nan],
    ["Estaciones finales", df_final["NGAsubSSN"].nunique(), np.nan],
    ["Test R2", m_test["R2"], np.nan],
    ["Test RMSE", m_test["RMSE"], np.nan],
    ["Pisco R2", m_pisco["R2"], np.nan],
    ["LOEO R2", m_loeo["R2"], np.nan],
    ["LOEO + purga estaciones R2", m_loeo_purge["R2"], np.nan],
], columns=["Indicador","Valor","Comentario"])

display(resumen_final)
export_csv(resumen_final,"10_resumen_final.csv")

print("\nMODELO FINAL")
print("Algoritmo: Gradient Boosting")
print("Variables: Mw + Rrup + Vs30")
print("Parámetros:", FINAL_PARAMS)
print("\nNotebook reproducible completado.")


## 11. Auditoría final de consistencia

Esta sección reúne las cifras que deben considerarse fuente computacional oficial para la redacción de la tesis. Ninguna de estas pruebas modifica el algoritmo, las variables ni los hiperparámetros ya congelados.


In [ ]:
auditoria_final = pd.DataFrame([
    ["Dataset final", len(df_final)],
    ["Eventos finales", df_final["NGAsubEQID"].nunique()],
    ["Estaciones finales", df_final["NGAsubSSN"].nunique()],
    ["Test R2", m_test["R2"]],
    ["Test RMSE", m_test["RMSE"]],
    ["Test MAE", m_test["MAE"]],
    ["Test BIAS", m_test["BIAS"]],
    ["Pisco R2", m_pisco["R2"]],
    ["Pisco purga estaciones R2", m_pisco_purged["R2"]],
    ["LOEO R2", m_loeo["R2"]],
    ["LOEO purga estaciones R2", m_loeo_purge["R2"]],
    ["SHAP Rrup", float(imp.loc[imp["Variable"]=="Rrup","mean_abs_SHAP"].iloc[0])],
    ["SHAP Mw", float(imp.loc[imp["Variable"]=="Mw","mean_abs_SHAP"].iloc[0])],
    ["SHAP Vs30", float(imp.loc[imp["Variable"]=="Vs30","mean_abs_SHAP"].iloc[0])],
], columns=["Indicador","Valor"])

display(auditoria_final)
export_csv(
    auditoria_final,
    "11_auditoria_resultados_oficiales.csv"
)

print("\nCONTROLES AUTOMÁTICOS")
assert len(df_final)==1987
assert df_final["NGAsubEQID"].nunique()==108
assert df_final["NGAsubSSN"].nunique()==692
assert round(m_test["R2"],4)==0.8728
assert round(m_test["RMSE"],4)==1.1516
assert round(m_pisco["R2"],4)==0.9521
assert round(m_loeo["R2"],4)==0.8725
assert round(m_loeo_purge["R2"],4)==0.8656

print("AUDITORÍA FINAL: OK")


## Reglas de interpretación para la tesis

- `R²` no es “porcentaje de aciertos”.
- El BIAS en escala logarítmica no equivale directamente a error porcentual individual.
- SHAP no demuestra causalidad.
- La envolvente empírica de error no es la sigma de un GMM.
- Pisco es un evento retenido fuera del desarrollo.
- La aplicación a Lima no es una validación local.
- `Sa_modelo / Sa_E.030` es una comparación numérica descriptiva, no un indicador de seguridad estructural.
